# SALM Fine-tuning 16 · Hearing the stumbles: stuttering event detection (SEP-28k)

> **Task** 3-s clip → which stuttering events occur (multi-label) · **Data** SEP-28k-E (podcasts)
> **Metric** macro-F1 over 5 events ↑ · **Result** 13.01% → **62.02%** (+377%) · **Time** ~1.5 h

Part of the [SALM fine-tuning series](README.md); see [00_Overview_and_Setup](00_Overview_and_Setup.ipynb) first.

## The problem: a transcriber is trained *not* to hear this

SEP-28k is 28,000 three-second clips from podcasts about stuttering, labeled by three annotators for five events:

| event | sounds like |
|---|---|
| prolongation | "Sssssnake" |
| block | a silent, stuck pause before a word |
| sound repetition | "b-b-b-ball" |
| word repetition | "I I I want" |
| interjection | "um", "uh", "like" |

An ASR model is trained to produce the *clean* transcript, smoothing all of this away. So the prompted base model never
reports interjections, one of the two most frequent events, and rarely gets the others right. An event counts as
present when at least 2 of 3 annotators marked it. The model lists events, comma-separated, or says `none`. The goal
is a model that hears and names these events on speakers it has never heard.

In [ ]:
import json, os, shlex, subprocess, sys, time
from pathlib import Path


def find_nemo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "examples" / "speechlm2" / "finetuning").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate the NeMo source tree from {start}. Open this notebook from inside the repo checkout.")


NEMO_ROOT = find_nemo_root(Path.cwd())
FT = NEMO_ROOT / "examples" / "speechlm2" / "finetuning"              # the tools used below
WORK = Path(os.environ.get("SALM_FT_WORK", NEMO_ROOT / "tutorials" / "speechlm2" / "finetuning" / "workdir")).absolute()
os.environ["SALM_FT_WORK"] = str(WORK)                                # data, manifests, experiments, exports
os.environ.pop("MPLBACKEND", None)  # the kernel's inline plotting backend would break matplotlib in child processes
if not os.environ.get("SALM_CHECKPOINT"):
    raise RuntimeError(
        "SALM_CHECKPOINT is not set. Point it at your SALM checkpoint: a directory in HuggingFace format "
        "exported by SALMAutomodel. Run `export SALM_CHECKPOINT=/path/to/checkpoint` before "
        "starting Jupyter, or set os.environ['SALM_CHECKPOINT'] in a cell above this one."
    )
CKPT = Path(os.environ["SALM_CHECKPOINT"]).expanduser().absolute()  # base SALM checkpoint (HuggingFace format)
if not CKPT.is_dir():
    raise RuntimeError(f"SALM_CHECKPOINT={CKPT} is not a directory. It must be a SALM checkpoint in HuggingFace format.")
PYTHON = os.environ.get("TRAIN_PYTHON", "python")                     # NeMo training environment
VLLM_PYTHON = os.environ.get("VLLM_PYTHON", "python")                 # vLLM 0.28 environment (ideally a separate one)
M, EXP, EXPORT, EVAL = (WORK / d for d in ("manifests", "experiments", "exports", "eval"))
q = shlex.quote                                                        # shell-quote prompts (keeps non-ASCII intact)
for d in (M, EXP, EXPORT, EVAL, WORK / "logs", WORK / "configs"):
    d.mkdir(parents=True, exist_ok=True)


RETRIES = int(os.environ.get("SALM_FT_RETRIES", 3))                   # raise when other jobs share the GPU


def sh(cmd, log=None, retries=1):
    """Run a shell command from the NeMo root. With `log`, output goes to a file and only its tail is shown."""
    for attempt in range(1, retries + 1):
        print(f"$ {cmd}" + (f"   > {log}" if log else ""), flush=True)
        if log:
            with open(WORK / "logs" / log, "w") as f:
                rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT, stdout=f, stderr=subprocess.STDOUT).returncode
            tail = (WORK / "logs" / log).read_text(errors="ignore").splitlines()
            print("\n".join(l for l in tail[-400:] if l.startswith(("[eval]", "[export]", "[scores]", "[merge]", "wrote")))[-3000:])
        else:
            rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT).returncode
        if rc == 0:
            return
        if attempt < retries:  # e.g. a vLLM engine starting while another job allocates GPU memory
            print(f"(attempt {attempt} failed, retrying in 90 s)")
            time.sleep(90)
    raise RuntimeError(f"command failed (rc={rc}): {cmd}" + (f"; see {WORK / 'logs' / log}" if log else ""))


def evaluate(model, manifest, name, task_args, extra=""):
    """Decode `manifest` with vLLM (greedy) and return the metric summary.

    Re-running a cell reuses a previous result only if its fingerprint (model, manifest, prompt, decoder and scoring
    settings, code revision) matches; a result produced from different inputs is an error, never silently reused.
    """
    out = EVAL / f"{name}.jsonl"
    sh(f"{VLLM_PYTHON} {FT}/vllm_task_eval.py --model {model} --manifest {manifest} --out {out} {task_args} {extra} "
       f"--reuse", log=f"{name}.log", retries=RETRIES)
    return json.load(open(f"{out}.summary.json"))


def tidy(keep, name):
    """Free disk: delete this tutorial's other merged exports (~63 GB each) and leftover training checkpoints."""
    import shutil

    for p in EXPORT.glob(f"{name}_*"):
        if p != keep:
            shutil.rmtree(p)
    shutil.rmtree(EXP / name / "checkpoints", ignore_errors=True)
    print(f"kept {keep}; removed the other {name} exports and checkpoints")


def samples(name, n=3, keys=("text", "pred_text")):
    """Print a few decoded examples."""
    rows = [json.loads(l) for l in open(EVAL / f"{name}.jsonl")]
    for r in rows[:n]:
        print("  |  ".join(f"{k}: {str(r.get(k))[:110]}" for k in keys))


def result(name, value, expected, tol):
    """Compare a number with the expected result, and record it in results.json."""
    ok = abs(value - expected) <= tol
    print(f"{'✅' if ok else '❌'} {name}: {value:.4f}  (expected: {expected} ± {tol})")
    res = WORK / "results.json"
    d = json.loads(res.read_text()) if res.exists() else {}
    d[name] = {"value": value, "expected": expected, "tolerance": tol, "ok": ok}
    res.write_text(json.dumps(d, indent=1))
    return ok


print("NeMo root:", NEMO_ROOT, "\nwork dir: ", WORK, "\ncheckpoint:", CKPT)

## 1. Data: three speakers is not a training set

The tutorial uses the speaker-disjoint "E" split. Its *train* part holds only **3 speakers** (the podcast hosts), so a
model trained on it learns three voices. The E-dev part has 141 speakers: 85% of them join training and 15% become the
dev set. Clips marked as music, no speech, poor audio or "unsure" are dropped.

In [ ]:
d = M / "sep28k"
sh(f"{PYTHON} {FT}/prepare_benchmarks.py sep28k", log="prep_sep28k.log")
sh(f"{PYTHON} {FT}/prepare_benchmarks.py librispeech", log="prep_ls.log")
for s in ("train", "dev", "test"):
    rows = [json.loads(l) for l in open(d / f"{s}.json")]
    print(f"{s:5s} {len(rows):6d} clips, {len({r['speaker'] for r in rows})} speakers, {sum(not r['events'] for r in rows) / len(rows):.0%} with no event")

## 2. Baseline

In [ ]:
TASK = "--task multilabel --max-tokens 32"
P2 = ("Does the speaker stutter in this clip? List the stuttering event types that occur (prolongation, block, sound "
      "repetition, word repetition, interjection), separated by commas, or answer none.")
base = {}
for p, extra in (("p1", ""), ("p2", f"--prompt {q(P2)}")):
    for s in ("dev", "test"):
        base[p, s] = evaluate(CKPT, d / f"{s}.json", f"stut_base_{p}_{s}", TASK, extra)
    print(f"{p}  dev macro-F1 {100 * base[p, 'dev']['score']:5.2f}%   test {100 * base[p, 'test']['score']:5.2f}%   "
          f"per event: {({k: round(v, 2) for k, v in base[p, 'test']['per_class_f1'].items()})}")
BEST_P = max(("p1", "p2"), key=lambda p: base[p, "dev"]["score"])  # prompt chosen on dev
BASELINE = 100 * base[BEST_P, "test"]["score"]  # its test score: the baseline fine-tuning is compared with
ORACLE = 100 * max(base[p, "test"]["score"] for p in ("p1", "p2"))  # best prompt chosen on test: an upper bound
print(f"baseline prompt (chosen on dev): {BEST_P}; test-oracle (best prompt on test): {ORACLE:.2f}")
print(f"\nbaseline = {BASELINE:.2f}%")
pairs = " ".join(f"{d}/{s}.json {d}/{s}_ft.json" for s in ("train", "dev", "test"))
sh(f"{PYTHON} {FT}/set_context.py --template {q(P2)} {pairs}")  # train with p2: it names the five event types

## 3. Config and training (~2.8 s/step, ~25 min)

In [ ]:
CFG = WORK / "configs" / "stut.yaml"
sh(f"{PYTHON} {FT}/make_ft_config.py --checkpoint {CKPT} --prompt manifest --out {CFG} --exp-dir {EXP / 'stut'} "
   f"--train-manifest {d}/train_ft.json --val-manifest {d}/dev_ft.json --val-name stut_dev "
   f"--train-encoder --train-proj --lr 1e-4 --warmup-steps 40 --max-steps 400 "
   f"--batch-tokens 24000 --limit-train-batches 50 --val-every-n-epochs 2 --limit-val-batches 50 --save-top-k -1")
sh(f"{PYTHON} {FT}/run_training.py --config {CFG} --preflight --smoke --keep-steps 200 400")

## 4. Select on dev

Expected dev macro-F1: **step 200 → 66.55%**, step 400 → 65.58%.

In [ ]:
dev = {}
for step in (200, 400):
    out = EXPORT / f"stut_s{step}"
    sh(f"{PYTHON} {FT}/export_checkpoint.py --exp-dir {EXP / 'stut'} --step {step} --base-checkpoint {CKPT} "
       f"--out {out} --delete-ckpt", log=f"export_stut_{step}.log")
    dev[step] = 100 * evaluate(out, d / "dev_ft.json", f"stut_s{step}_dev", TASK)["score"]
    print(f"step {step}: dev macro-F1 {dev[step]:.2f}%")
BEST = max(dev, key=dev.get)

## 5. Test, once

In [ ]:
MODEL = EXPORT / f"stut_s{BEST}"
ft = evaluate(MODEL, d / "test_ft.json", "stut_TEST", TASK)
guard = evaluate(MODEL, M / "librispeech" / "test_clean.json", "stut_librispeech", "--task asr --normalizer english")
FT_F1 = 100 * ft["score"]
print(f"test macro-F1: base {BASELINE:.2f}% → fine-tuned {FT_F1:.2f}% ({(FT_F1 - BASELINE) / BASELINE:+.1%}); LibriSpeech {100 * guard['score']:.2f}%")
print("per event:", {k: round(v, 2) for k, v in ft["per_class_f1"].items()})
samples("stut_TEST", n=5)
result("16 baseline test macro-F1", BASELINE, 13.01, 0.5)
result("16 fine-tuned test macro-F1", FT_F1, 62.02, 2.0)

Expected results:

| F1 on test | prolong. | block | sound rep. | word rep. | interjection | **macro** | LibriSpeech WER |
|---|---:|---:|---:|---:|---:|---:|---:|
| base, prompt chosen on dev (p1) | 0.18 | 0.04 | 0.27 | 0.16 | 0.00 | 13.01% | 1.54% |
| **fine-tuned** | 0.47 | 0.42 | 0.51 | 0.85 | 0.85 | **62.02%** | 1.69% |

**+377% relative.** The prompt chosen on dev (p1) is also the test-oracle. Interjections and word repetitions reach
0.85, at or above per-class results published for this dataset. Blocks stay hardest: they are mostly *silence*, so
there's little to hear.

## Tidy up

Keep the selected model; delete the other exports (~63 GB each) and any leftover training checkpoints.

In [ ]:
tidy(keep=MODEL, name="stut")

## Takeaways

1. **Read the split's speaker counts.** A 3-speaker training set is a trap, even with 12,000 clips.
2. **Multi-label as text works.** A comma-separated list plus a lenient parser.
3. **Fine-tuning can re-open what pretraining taught the model to ignore** (disfluencies).